# 05 · Build the presentation (consulting deck)

This notebook **is** the presentation. Every sentence, number and chart dataset on the Claude artifact (`docs/story.html`) is written here, one section per slide, and previewed below each cell as Markdown so you can read and debug it without opening the page.

```
notebooks 01–04  ──►  docs/results.json  ──►  this notebook (05)  ──►  docs/story.html  ──►  Claude artifact
```

- **Numbers** come only from `docs/results.json` (notebooks 01–04) and `reference/assumptions.csv` (sourced outside inputs).
- **Wording** lives in the code cells as plain strings with light Markdown (`**bold**`, `*italic*`, `[link](url)`).
- **`site/story_template.html`** only lays out and draws; it has no text or numbers of its own.

| Slide | Time | Feeds from |
|---|---|---|
| Opening + executive summary | 0:00 – 1:30 | everything |
| 1 · Situation: the drugs, and who decides access | 1:30 – 2:45 | outside sources, notebook 03 |
| 2 · The client and the brief (problem statement, scope) | 2:45 – 3:45 | outside sources |
| 3 · Approach | 3:45 – 4:15 | — |
| 4 · Data | 4:15 – 4:45 | notebook 01 |
| 5 · A · Who keeps coming back | 4:45 – 6:30 | notebook 02 |
| 6 · B · The drugs and their price | 6:30 – 8:00 | notebook 03 |
| 7 · C · Where it pays for itself | 8:00 – 10:00 | notebook 04 |
| 8 · C · The rule a payer can write | 10:00 – 12:00 | notebook 04 |
| 9 · Recommendation | 12:00 – 13:45 | notebooks 02–04 |
| 10 · Risks and next steps | 13:45 – 15:00 | all |

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import html, json, re
import pandas as pd
from IPython.display import Markdown, display
from src.config import ROOT, RESULTS

R = json.loads(RESULTS.read_text())
C, P, M = R["cleaning"], R["profile"], R["models"]         # notebooks 01-02
V, L = R["value"], R["lever"]                               # notebooks 04 and 03
print("Sections available:", list(R))

Sections available: ['cleaning', 'sources', 'profile', 'models', 'lever', 'value']


## Helpers

`md()` turns the light Markdown used here into HTML for the page. `preview()` shows each section the way it reads on the page.

In [2]:
def md(s: str) -> str:
    """Inline Markdown -> HTML: **bold**, *italic*, `code`, [text](url)."""
    s = html.escape(s, quote=False)
    s = re.sub(r"\[([^\]]+)\]\(([^)]+)\)", r'<a href="\2">\1</a>', s)
    s = re.sub(r"\*\*(.+?)\*\*", r"<b>\1</b>", s)
    s = re.sub(r"(?<![*\w])\*(?!\s)(.+?)(?<!\s)\*(?![*\w])", r"<i>\1</i>", s)
    s = re.sub(r"`(.+?)`", r'<span class="mono">\1</span>', s)
    return s

def pct(v, d=1): return f"{v:,.{d}f}%"
def num(v, d=0): return f"{v:,.{d}f}"

# Block constructors. Each keeps the Markdown source (for preview) and the HTML (for the page).
def p(text, cls=None):    return {"type": "p", "md": text, "html": md(text), **({"cls": cls} if cls else {})}
def h3(text):             return {"type": "h3", "md": text, "html": md(text)}
def say(text):            return {"type": "say", "md": text, "html": md(text)}
def bullets(items, title=None): return {"type": "list", "title": title, "md": items, "items": [md(i) for i in items]}
def text(title, body):    return {"type": "text", "title": title, "md": body, "html": md(body)}
def tiles(items):         return {"type": "tiles", "items": [{"value": v, "label": md(l), "src": s} for v, l, s in items]}
def cols(*blocks):        return {"type": "cols", "blocks": list(blocks)}
def table(columns, rows, highlight=None):
    """columns: list of (label, is_numeric)."""
    return {"type": "table", "columns": [{"label": c, "num": n} for c, n in columns], "rows": rows,
            **({"highlight": highlight} if highlight is not None else {})}
def chart(kind, title, data, opts, sub=None, legend=None):
    return {"type": "chart", "kind": kind, "title": title, "sub": sub, "legend": legend, "data": data, "opts": opts}

def preview(sec):
    """Render one section as Markdown so it can be read and checked inside the notebook."""
    out = [f"## {sec['title']}  \n*{sec['time']}*"]
    def walk(b):
        t = b["type"]
        if t in ("p", "h3"):   out.append(("### " if t == "h3" else "") + b["md"])
        elif t == "say":       out.append("> **Say:** " + b["md"])
        elif t == "callout":   out.append(f"> **{b.get('title') or ''}** " + b["md"])
        elif t == "text":      out.append(f"**{b['title']}**  \n{b['md']}")
        elif t == "list":      out.append(("**" + b["title"] + "**\n\n" if b.get("title") else "") + "\n".join("- " + i for i in b["md"]))
        elif t == "tiles":     out.append("\n".join(f"- **{i['value']}** {html.unescape(re.sub('<[^>]+>', '', i['label']))}" + (f" *({i['src']})*" if i.get("src") else "") for i in b["items"]))
        elif t == "cols":      [walk(x) for x in b["blocks"]]
        elif t == "table":     out.append(pd.DataFrame([[re.sub("<[^>]+>", "", str(v)) for v in r] for r in b["rows"]], columns=[c["label"] for c in b["columns"]]).to_markdown(index=False))
        elif t == "chart":
            out.append(f"**Chart · {b['title']}**" + (f" — *{b['sub']}*" if b.get("sub") else ""))
            d = b["data"]
            if b["kind"] in ("hbars",):     out.append(pd.DataFrame([{"label": r["label"], "value": r["value"]} for r in d]).to_markdown(index=False))
            elif b["kind"] == "pairbars":   out.append(pd.DataFrame([{"label": r["label"], **dict(zip(b["opts"]["names"], r["values"]))} for r in d]).to_markdown(index=False))
            elif b["kind"] == "lines":      out.append(pd.DataFrame({s["name"]: dict(s["pts"]) for s in d}).round(1).to_markdown())
            elif b["kind"] == "hgroup":     out.append(pd.DataFrame([{"group": r["label"], **dict(zip(b["opts"]["names"], r["values"]))} for r in d]).round(0).to_markdown(index=False))
            elif b["kind"] == "tornado":    out.append(pd.DataFrame([{"assumption": r["label"], "low": r["low"], "high": r["high"]} for r in d]).round(0).to_markdown(index=False))
            elif b["kind"] == "scatter":    out.append(pd.DataFrame([{"label": q["label"], "x": q["x"], "y": q["y"]} for q in d]).to_markdown(index=False))
        elif t == "issues":    out.append("\n".join(f"- **{q['k']}. {q['q']}** — {re.sub('<[^>]+>', '', q['how'])}" + (f" **[{q['verdict']}]**" if q.get("verdict") else "") for q in b["items"]))
        elif t == "refs":      out.append("Sources:\n" + "\n".join(f"{i+1}. {re.sub('<[^>]+>', '', r)}" for i, r in enumerate(b["items"])))
        elif t == "flow":      out.append("\n".join(f"- **{n['title']}** ({n['k']}): {n['text']}" for n in b["nodes"]) + "\n\n" + "\n".join(f"- *{j['key']}*: {j['text']}" for j in b["joins"]))
        elif t == "viewer":    out.append("Data viewer tabs: " + ", ".join(t_["label"] for t_ in b["tabs"]))
        elif t == "privacy":   out.append(f"**{b['title']}**  \n{b['md']}"); walk(b["table"]); out.append("\n".join("- " + x for x in b["bullets_md"]))
        elif t == "reco":      out.append("\n".join(f"{i+1}. **{r['k']}** — {r['md']}" for i, r in enumerate(b["items"])))
        elif t == "limits":    out.append("\n\n".join(f"**{c['title']}**\n" + "\n".join("- " + x for x in c["md"]) for c in b["cols"]))
    for b in sec["blocks"]:
        walk(b)
    display(Markdown("\n\n".join(out)))

SECTIONS = []

def callout(text, title=None, plain=False):
    return {"type": "callout", "title": title, "md": text, "html": md(text), **({"plain": True} if plain else {})}
def money(v, d=0):
    return ("−" if v < 0 else "") + "$" + f"{abs(v):,.{d}f}"

## Key numbers used across slides

Pulled once so every slide quotes the same value.

In [3]:
SG_N, SG_G, GL_N = "SGLT2 · negotiated (2026)", "SGLT2 · gross 2024", "GLP-1 · negotiated (2027)"
T = V["target"]
rule = {r["scenario"]: r for r in V["rule"]}
tiers = {t["stays_group"]: t for t in V["tiers"]}
groups = {g["stays_group"]: g for g in V["groups"]}
trend = {t["year"]: t for t in L["trend"]}
prior = {r["level"]: r for r in P["by"]["Prior hospital stays (past year)"]}
disc = {r["level"]: r for r in P["by"]["Discharged to"]}
best = max(M["comparison"], key=lambda r: r["ROC-AUC"])
S2 = V["stress"]["2+ prior stays"]
KEY = {
    "target_pct": T["pct"], "target_n": T["n"], "target_r30": T["readmit_30"], "rest_r30": T["readmit_30_rest"], "target_later": T["later_readmit"],
    "net": V["base_net"], "one_prior_net": T["one_prior_net"], "net_1000": rule[SG_N]["net / yr ($)"], "avoided_1000": rule[SG_N]["admissions avoided / yr"],
    "be": V["breakeven"][SG_N], "be_gross": V["breakeven"][SG_G], "cost": V["cost"]["base"],
    "price_2": tiers["2 prior stays"]["breakeven_price_month"], "price_3": tiers["3+ prior stays"]["breakeven_price_month"], "price_1": tiers["1 prior stay"]["breakeven_price_month"],
    "neg_month": L["annual_cost"][SG_N] / 12, "gross_month": L["annual_cost"][SG_G] / 12,
    "s_half": S2["future stays = 50% of past"], "s_low_cost": S2["cost of a stay: diabetes/heart-failure, 2024 $"], "s_effect": S2["effect 4% (trial low end)"],
    "s_gross": S2["2024 gross price"], "s_combined": S2["combined downside (50% future + low cost)"],
    "medicare_net": V["stress"]["2+ prior stays, Medicare-insured only"]["base"], "heart_net": V["stress"]["2+ prior stays, heart diagnosis only"]["base"],
    "glp1_2": V["glp1_needed"]["2+ prior stays"], "newer_claims_13": trend[2013]["newer_share_claims"], "newer_claims_24": trend[2024]["newer_share_claims"],
    "newer_cost_24": trend[2024]["newer_share_cost"], "spend_24": trend[2024]["cost_B"], "n_api": sum(s["kind"] == "API" for s in R["sources"]),
}
pd.Series(KEY).round(2).to_frame("value")

,value
target_pct,3.40
target_n,2397.00
target_r30,21.50
rest_r30,8.50
target_later,71.10
net,2766.00
one_prior_net,-484.00
net_1000,94720.00
avoided_1000,10.00
be,1.26


## Opening and executive summary (0:00 – 1:30)

The opener is personal; **rewrite it in your own words.** Then the answer first.

In [4]:
HERO = {
    "eyebrow": "Case study · framed as a pharma market-access engagement · Adhvaith Vijay",
    "title": "For which patients does a GLP-1 or SGLT2 drug pay for itself?",
    "lede": "A value case for a diabetes-drug manufacturer: which patients should insurers make it easy to get these drugs, because the hospital stays they prevent cover the cost?",
    "opener": '"My dad has type 2 diabetes. The newer drugs, like Jardiance and Ozempic, don\'t just lower blood sugar. In large trials they also kept patients out of the hospital. But they cost hundreds of dollars a month, so insurers ask whether they\'re worth it. I wanted to find out: for which patients do the hospital stays these drugs prevent cover what the drug costs?"',
    "opener_note": "Opening, in my own words. About 30 seconds.",
}
SECTIONS = []
summary = {"id": "summary", "nav": "Summary", "title": "The answer: make the case to payers, for patients with two or more recent hospital stays", "time": "0:30 – 1:30", "blocks": [
    callout(f"In the US, **the payer, not the doctor, decides who can actually get these drugs**, so the case has to be made to payers. At Medicare's negotiated price, covering an SGLT2 drug **pays for itself through avoided hospital stays for diabetic patients with two or more hospital stays in the past year** ({pct(KEY['target_pct'])} of diabetic inpatients): about {money(KEY['net'])} per patient-year. With one prior stay it doesn't quite pay; with none, it's a clinical choice, not a saving.", "The answer"),
    bullets([
        f"**Who.** {num(KEY['target_n'])} of {num(C['patients'])} diabetic inpatients had 2+ other stays in the past year. They're readmitted within 30 days at {pct(KEY['target_r30'])} (vs. {pct(KEY['rest_r30'])}), and {pct(KEY['target_later'], 0)} came back at some point after discharge.",
        f"**What it's worth.** A hospital stay costs Medicare about {money(KEY['cost'])} (2026 dollars). Per 1,000 diabetic discharges, covering this group avoids about {KEY['avoided_1000']:.0f} admissions a year, a net saving of about {money(round(KEY['net_1000'], -3))}. Covering pays for itself up to about {money(KEY['price_2'])} a month for patients with 2 prior stays and {money(KEY['price_3'])} with 3+.",
        f"**What to do.** Bring this to Medicare Advantage plans and ACOs, which pay for both the drug and the hospital stay: propose no prior authorization or step therapy for this group, a net price per tier, and a rebate if admissions don't fall.",
    ], "Three supporting points"),
    say("I'll start with the answer, then show how I got there: who decides access, who these patients are, what the drugs cost, and where the math works."),
]}
SECTIONS.append(summary); preview(summary)

## The answer: make the case to payers, for patients with two or more recent hospital stays  
*0:30 – 1:30*

> **The answer** In the US, **the payer, not the doctor, decides who can actually get these drugs**, so the case has to be made to payers. At Medicare's negotiated price, covering an SGLT2 drug **pays for itself through avoided hospital stays for diabetic patients with two or more hospital stays in the past year** (3.4% of diabetic inpatients): about $2,766 per patient-year. With one prior stay it doesn't quite pay; with none, it's a clinical choice, not a saving.

**Three supporting points**

- **Who.** 2,397 of 69,987 diabetic inpatients had 2+ other stays in the past year. They're readmitted within 30 days at 21.5% (vs. 8.5%), and 71% came back at some point after discharge.
- **What it's worth.** A hospital stay costs Medicare about $17,094 (2026 dollars). Per 1,000 diabetic discharges, covering this group avoids about 10 admissions a year, a net saving of about $95,000. Covering pays for itself up to about $313 a month for patients with 2 prior stays and $619 with 3+.
- **What to do.** Bring this to Medicare Advantage plans and ACOs, which pay for both the drug and the hospital stay: propose no prior authorization or step therapy for this group, a net price per tier, and a rebate if admissions don't fall.

> **Say:** I'll start with the answer, then show how I got there: who decides access, who these patients are, what the drugs cost, and where the math works.

## Slide 1 · Situation: who decides access → problem statement (1:30 – 3:00)

The access table is the crucial framing: the payer, not the doctor, controls who gets the drug.

In [5]:
REFS = [
    "[CDC, National Diabetes Statistics Report: At a glance](https://www.cdc.gov/diabetes/php/data-research/index.html)",
    "[American Diabetes Association, Economic Costs of Diabetes in the U.S. in 2022](https://diabetes.org/about-us/statistics/cost-diabetes) (Parker et al., *Diabetes Care*)",
    "[Rubin & Shah, \"Predicting and Preventing Acute Care Re-Utilization by Patients with Diabetes,\" *Current Diabetes Reports*, 2021](https://pmc.ncbi.nlm.nih.gov/articles/PMC8418292)",
    "American Diabetes Association, Standards of Care in Diabetes—2025, Section 9: Pharmacologic Approaches (*Diabetes Care* 48, Supplement 1)",
    "[GLP-1 RA / SGLT2i use after new ASCVD diagnosis, 2018–2022, *BMJ Open Diabetes Research & Care*, 2024](https://doaj.org/article/241fc9acf8aa4641a972e8d294236447)",
    "[CMS, Medicare negotiated prices for 2026](https://cms.gov/files/document/fact-sheet-negotiated-prices-initial-price-applicability-year-2026.pdf) and [2027 (semaglutide)](https://www.healthcaredive.com/news/medicare-price-negotiation-wegovy-ozempic-trelegy-2027/806578/)",
]
situation = {"id": "situation", "nav": "1 · Situation", "title": "1 · Effective, expensive drugs, and the payer decides who gets them", "time": "1:30 – 2:45", "blocks": [
    tiles([
        ("38.4M", "Americans have diabetes (11.6% of the population)", "CDC, 2021 data [1]"),
        ("$412.9B", "annual US cost of diabetes, $306.6B of it direct medical care", "ADA, 2022 [2]"),
        ("16–20%", "of hospitalized diabetic patients are readmitted within 30 days, about $20–25B a year", "Rubin & Shah, 2021 [3]"),
        (f"${KEY['spend_24']:.1f}B", "Medicare Part D gross spending on diabetes drugs in 2024, from my CMS pull", "CMS API, notebook 03"),
    ]),
    h3("Where these drugs sit in treatment"),
    table([("Step", False), ("Drugs", False), ("Cost", False), ("What it does", False)], [
        ["<b>1 · First line</b>", "Metformin (plus diet and exercise)", "Cheap generic, a few dollars a month", "Lowers blood sugar; the default starting drug for most patients"],
        ["<b>2 · Add-ons</b>", "Sulfonylureas, DPP-4 inhibitors, pioglitazone", "Mostly generic and inexpensive", "More blood-sugar control when metformin isn't enough"],
        ["<b>3 · Insulin</b>", "Long- and short-acting insulins", "Varies; capped at $35 a month for Medicare patients", "For advanced disease or very high blood sugar"],
        ["<b>The newer classes</b>", "<b>GLP-1</b> (Ozempic, Trulicity, Mounjaro) and <b>SGLT2</b> (Jardiance, Farxiga)", f"Hundreds of dollars a month (Jardiance $541 gross in 2024, $197 negotiated)", "Lower blood sugar <b>and</b>, in large trials, cut heart attacks, heart-failure hospitalizations and, for SGLT2, hospital stays of any kind"],
    ]),
    cols(
        text("What changed in the guidelines", "The traditional ladder treats GLP-1 and SGLT2 drugs as the “big guns” added after metformin and other pills. For patients with **heart disease, heart failure or kidney disease**, guidelines now recommend them **early, regardless of blood sugar and whether the patient is already on metformin** [4]. For those patients, they're no longer a last resort."),
        text("The catch", f"They're {pct(KEY['newer_cost_24'], 0)} of Medicare diabetes drug spending but only {pct(KEY['newer_claims_24'], 0)} of prescriptions filled, and uptake among eligible patients is still low [5]. Plans often still apply the old ladder through **step therapy**: try metformin and cheaper drugs first. Medicare's negotiated prices (Jardiance $197 a month from 2026, semaglutide $274 from 2027) change that math [6]."),
    ),
    h3("Who actually decides whether a patient gets the drug"),
    table([("Who", False), ("What they control", False)], [
        ["<b>Doctor</b>", "Chooses the drug and writes the prescription."],
        ["<b>Payer</b> (insurance plan)", "Decides whether the drug is covered (formulary), whether the doctor must file a <b>prior authorization</b>, whether the patient must fail cheaper drugs first (<b>step therapy</b>), and the patient's copay."],
        ["<b>Patient</b>", "Fills the prescription only if it's covered and affordable."],
    ]),
    callout("In the US, **who can actually get a GLP-1 or SGLT2 drug is decided by the payer, not the doctor.** A doctor can prescribe it, but if the plan requires paperwork, cheaper drugs first, or a high copay, many patients never start it. Payers loosen those rules only where they believe the drug pays for itself.", "Why the payer is the audience"),
    say("A manufacturer can't make a doctor prescribe. What it can do is show payers where removing the access hurdles pays for itself."),
    {"type": "refs", "items": [md(r) for r in REFS]},
]}
SECTIONS.append(situation); preview(situation)

## 1 · Effective, expensive drugs, and the payer decides who gets them  
*1:30 – 2:45*

- **38.4M** Americans have diabetes (11.6% of the population) *(CDC, 2021 data [1])*
- **$412.9B** annual US cost of diabetes, $306.6B of it direct medical care *(ADA, 2022 [2])*
- **16–20%** of hospitalized diabetic patients are readmitted within 30 days, about $20–25B a year *(Rubin & Shah, 2021 [3])*
- **$59.9B** Medicare Part D gross spending on diabetes drugs in 2024, from my CMS pull *(CMS API, notebook 03)*

### Where these drugs sit in treatment

| Step              | Drugs                                                               | Cost                                                                        | What it does                                                                                                                         |
|:------------------|:--------------------------------------------------------------------|:----------------------------------------------------------------------------|:-------------------------------------------------------------------------------------------------------------------------------------|
| 1 · First line    | Metformin (plus diet and exercise)                                  | Cheap generic, a few dollars a month                                        | Lowers blood sugar; the default starting drug for most patients                                                                      |
| 2 · Add-ons       | Sulfonylureas, DPP-4 inhibitors, pioglitazone                       | Mostly generic and inexpensive                                              | More blood-sugar control when metformin isn't enough                                                                                 |
| 3 · Insulin       | Long- and short-acting insulins                                     | Varies; capped at $35 a month for Medicare patients                         | For advanced disease or very high blood sugar                                                                                        |
| The newer classes | GLP-1 (Ozempic, Trulicity, Mounjaro) and SGLT2 (Jardiance, Farxiga) | Hundreds of dollars a month (Jardiance $541 gross in 2024, $197 negotiated) | Lower blood sugar and, in large trials, cut heart attacks, heart-failure hospitalizations and, for SGLT2, hospital stays of any kind |

**What changed in the guidelines**  
The traditional ladder treats GLP-1 and SGLT2 drugs as the “big guns” added after metformin and other pills. For patients with **heart disease, heart failure or kidney disease**, guidelines now recommend them **early, regardless of blood sugar and whether the patient is already on metformin** [4]. For those patients, they're no longer a last resort.

**The catch**  
They're 76% of Medicare diabetes drug spending but only 33% of prescriptions filled, and uptake among eligible patients is still low [5]. Plans often still apply the old ladder through **step therapy**: try metformin and cheaper drugs first. Medicare's negotiated prices (Jardiance $197 a month from 2026, semaglutide $274 from 2027) change that math [6].

### Who actually decides whether a patient gets the drug

| Who                    | What they control                                                                                                                                                                               |
|:-----------------------|:------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| Doctor                 | Chooses the drug and writes the prescription.                                                                                                                                                   |
| Payer (insurance plan) | Decides whether the drug is covered (formulary), whether the doctor must file a prior authorization, whether the patient must fail cheaper drugs first (step therapy), and the patient's copay. |
| Patient                | Fills the prescription only if it's covered and affordable.                                                                                                                                     |

> **Why the payer is the audience** In the US, **who can actually get a GLP-1 or SGLT2 drug is decided by the payer, not the doctor.** A doctor can prescribe it, but if the plan requires paperwork, cheaper drugs first, or a high copay, many patients never start it. Payers loosen those rules only where they believe the drug pays for itself.

> **Say:** A manufacturer can't make a doctor prescribe. What it can do is show payers where removing the access hurdles pays for itself.

Sources:
1. CDC, National Diabetes Statistics Report: At a glance
2. American Diabetes Association, Economic Costs of Diabetes in the U.S. in 2022 (Parker et al., Diabetes Care)
3. Rubin &amp; Shah, "Predicting and Preventing Acute Care Re-Utilization by Patients with Diabetes," Current Diabetes Reports, 2021
4. American Diabetes Association, Standards of Care in Diabetes—2025, Section 9: Pharmacologic Approaches (Diabetes Care 48, Supplement 1)
5. GLP-1 RA / SGLT2i use after new ASCVD diagnosis, 2018–2022, BMJ Open Diabetes Research &amp; Care, 2024
6. CMS, Medicare negotiated prices for 2026 and 2027 (semaglutide)

## Slide 2 · The client and the brief (2:45 – 3:45)

Who the client is, what's changing for them, the decision they face, and the problem statement. The client is hypothetical; the market facts are real.

In [6]:
brief = {"id": "brief", "nav": "2 · Client", "title": "2 · The client and the brief", "time": "2:45 – 3:45", "blocks": [
    callout("**A manufacturer of an SGLT2 diabetes drug (hypothetical), specifically its US market access and health economics team**: the people who negotiate with insurance plans over coverage, price and rebates.", "The client"),
    table([("For the client", False), ("", False)], [
        ["<b>What's changing</b>", f"Medicare's negotiated price takes effect in 2026: Jardiance drops to $197 a month (from a 2023 list price of $573). Revenue per prescription falls by about two-thirds, so growth has to come from <b>more appropriate patients actually getting the drug</b>."],
        ["<b>Their barrier</b>", "Doctors already have the evidence and the guidelines. What stops many patients is the plan's access rules: prior authorization, step therapy and copays."],
        ["<b>The competition</b>", f"GLP-1 drugs are the other major newer class and reach their own negotiated price in 2027, so plans are re-examining what they cover and on what terms."],
        ["<b>The decision</b>", "Which patient group to lead plan negotiations with, what coverage terms to ask for, and what net price or rebate to offer in return."],
    ]),
    callout("**For which diabetic patients does covering an SGLT2 drug (and, by comparison, a GLP-1) pay for itself for the payer through the hospital stays it prevents, at what net price, and what coverage rule should the manufacturer propose to Medicare Advantage plans and ACOs?**", "Problem statement"),
    table([("Scope", False), ("", False)], [
        ["<b>In</b>", "Medicare-age diabetic patients; SGLT2 at 2024 gross vs. 2026 negotiated price; GLP-1 as a comparison; savings from avoided hospital stays."],
        ["<b>Out</b>", "Commercial plans; benefits other than hospital stays (deaths, kidney failure, quality of life); targeting individual doctors."],
        ["<b>A good answer</b>", "A coverage rule a plan can run in its own claims data, the net price at which it pays for itself, and a pilot to prove it."],
    ]),
    say("Here's the client and the decision they face. Everything that follows is built to answer one question: which patients, at what price, and with what coverage rule."),
]}
SECTIONS.append(brief); preview(brief)

## 2 · The client and the brief  
*2:45 – 3:45*

> **The client** **A manufacturer of an SGLT2 diabetes drug (hypothetical), specifically its US market access and health economics team**: the people who negotiate with insurance plans over coverage, price and rebates.

| For the client   |                                                                                                                                                                                                                                                         |
|:-----------------|:--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| What's changing  | Medicare's negotiated price takes effect in 2026: Jardiance drops to $197 a month (from a 2023 list price of $573). Revenue per prescription falls by about two-thirds, so growth has to come from more appropriate patients actually getting the drug. |
| Their barrier    | Doctors already have the evidence and the guidelines. What stops many patients is the plan's access rules: prior authorization, step therapy and copays.                                                                                                |
| The competition  | GLP-1 drugs are the other major newer class and reach their own negotiated price in 2027, so plans are re-examining what they cover and on what terms.                                                                                                  |
| The decision     | Which patient group to lead plan negotiations with, what coverage terms to ask for, and what net price or rebate to offer in return.                                                                                                                    |

> **Problem statement** **For which diabetic patients does covering an SGLT2 drug (and, by comparison, a GLP-1) pay for itself for the payer through the hospital stays it prevents, at what net price, and what coverage rule should the manufacturer propose to Medicare Advantage plans and ACOs?**

| Scope         |                                                                                                                                            |
|:--------------|:-------------------------------------------------------------------------------------------------------------------------------------------|
| In            | Medicare-age diabetic patients; SGLT2 at 2024 gross vs. 2026 negotiated price; GLP-1 as a comparison; savings from avoided hospital stays. |
| Out           | Commercial plans; benefits other than hospital stays (deaths, kidney failure, quality of life); targeting individual doctors.              |
| A good answer | A coverage rule a plan can run in its own claims data, the net price at which it pays for itself, and a pilot to prove it.                 |

> **Say:** Here's the client and the decision they face. Everything that follows is built to answer one question: which patients, at what price, and with what coverage rule.

## Slide 3 · Approach (3:45 – 4:15)

In [7]:
approach = {"id": "approach", "nav": "3 · Approach", "title": "3 · Three questions, each feeding the next", "time": "3:45 – 4:15", "blocks": [
    {"type": "issues", "items": [
        {"k": "A", "q": "Which diabetic patients keep coming back to hospital?",
         "how": md("**Hypothesis:** a small, identifiable group drives repeat admissions. **Data:** 101,766 hospital stays (Kaggle, 1999–2008). **Methods:** SQL profiling; penalized logistic regression, random forest and a neural network."),
         "where": "Output → each patient's prior hospital stays, and what happened after discharge", "verdict": "Confirmed"},
        {"k": "B", "q": "What do the drugs cost the payer?",
         "how": md("**Hypothesis:** price is the payer's objection. **Data:** CMS Part D 2013–2024 (my API pulls), CMS negotiated prices. **Methods:** same-source trend; cost per patient-year, gross vs. negotiated."),
         "where": "Output → annual cost of treatment", "verdict": "Confirmed"},
        {"k": "C", "q": "Where does covering the drug pay for itself, and what coverage rule follows?",
         "how": md("**Hypothesis:** at negotiated prices, only for frequent hospital users. **Data:** A's prior stays × B's prices × trial effect (EMPA-REG) × Medicare's payment per stay (CMS, in 2026 dollars). **Methods:** break-even model, check against what happened after discharge, stress tests, price per tier."),
         "where": "Output → a coverage rule and the price at which it pays", "verdict": "Confirmed, narrower than expected"},
    ]},
    say("Each part feeds the next: A tells us how often each patient was in hospital, B what a year of the drug costs, and C combines them with the trial effect into a rule a payer can write."),
]}
SECTIONS.append(approach); preview(approach)

## 3 · Three questions, each feeding the next  
*3:45 – 4:15*

- **A. Which diabetic patients keep coming back to hospital?** — Hypothesis: a small, identifiable group drives repeat admissions. Data: 101,766 hospital stays (Kaggle, 1999–2008). Methods: SQL profiling; penalized logistic regression, random forest and a neural network. **[Confirmed]**
- **B. What do the drugs cost the payer?** — Hypothesis: price is the payer's objection. Data: CMS Part D 2013–2024 (my API pulls), CMS negotiated prices. Methods: same-source trend; cost per patient-year, gross vs. negotiated. **[Confirmed]**
- **C. Where does covering the drug pay for itself, and what coverage rule follows?** — Hypothesis: at negotiated prices, only for frequent hospital users. Data: A's prior stays × B's prices × trial effect (EMPA-REG) × Medicare's payment per stay (CMS, in 2026 dollars). Methods: break-even model, check against what happened after discharge, stress tests, price per tier. **[Confirmed, narrower than expected]**

> **Say:** Each part feeds the next: A tells us how often each patient was in hospital, B what a year of the drug costs, and C combines them with the trial effect into a rule a payer can write.

## Slide 4 · Data (4:15 – 4:45)

One table and one privacy line. Sample rows and the full privacy evidence are in the appendix.

In [8]:
WHAT = {"Kaggle": "Patients, prior stays, drugs and readmissions, 1999–2008",
        "national": "National prescriptions, fills and cost per drug, every year 2013–2024",
        "Inpatient": "What a hospital stay costs Medicare, by type of stay (2024)",
        "BLS": "Hospital price index, to put every dollar in 2026",
        "RxClass": "Drug → class mapping (ATC), shared by all notebooks"}
src_rows = [[s["name"], f'<span class="chip">{"My API pull" if s["kind"] == "API" else "Downloaded"}</span>', num(s["rows"]),
             next(v for k, v in WHAT.items() if k in s["name"])] for s in R["sources"]]
data = {"id": "data", "nav": "4 · Data", "title": f"4 · One downloaded dataset, {KEY['n_api']} API pulls, no patient identifiers", "time": "4:15 – 4:45", "blocks": [
    table([("Source", False), ("How I got it", False), ("Rows", True), ("What it answers", False)], src_rows),
    p("Outside inputs with sources (trial effect, negotiated prices, benchmarks) are in `reference/assumptions.csv`. **No patient-identifying information anywhere:** the hospital data is a de-identified research extract, and every API source is a national total. Sample rows and the privacy checks are in notebook 01."),
]}
SECTIONS.append(data); preview(data)

## 4 · One downloaded dataset, 4 API pulls, no patient identifiers  
*4:15 – 4:45*

| Source                                                          | How I got it   |    Rows | What it answers                                                       |
|:----------------------------------------------------------------|:---------------|--------:|:----------------------------------------------------------------------|
| Kaggle · Diabetes 130-US Hospitals (1999–2008)                  | Downloaded     | 101,766 | Patients, prior stays, drugs and readmissions, 1999–2008              |
| CMS Part D Prescribers by Geography & Drug, national, 2013–2024 | My API pull    |   1,404 | National prescriptions, fills and cost per drug, every year 2013–2024 |
| CMS Medicare Inpatient Hospitals by Geography & Service (2024)  | My API pull    |     767 | What a hospital stay costs Medicare, by type of stay (2024)           |
| BLS CPI, inpatient hospital services (2017–2026)                | My API pull    |     115 | Hospital price index, to put every dollar in 2026                     |
| NLM RxClass (ATC classes)                                       | My API pull    |      48 | Drug → class mapping (ATC), shared by all notebooks                   |

Outside inputs with sources (trial effect, negotiated prices, benchmarks) are in `reference/assumptions.csv`. **No patient-identifying information anywhere:** the hospital data is a de-identified research extract, and every API source is a national total. Sample rows and the privacy checks are in notebook 01.

## Slide 5 · A · Who keeps coming back (4:45 – 6:30)

Notebook 02.

In [9]:
p0, p3, home, snf = prior["0"], prior["3+"], disc["Home"], disc["Skilled nursing / care facility"]
nn = next(r for r in M["comparison"] if r["model"].startswith("Neural"))
partA = {"id": "partA", "nav": "5 · A: Who", "title": f"5 · A: Prior hospital stays are the clearest signal: {pct(p3['readmit_pct'])} readmitted with 3+ vs. {pct(p0['readmit_pct'])} with none", "time": "4:45 – 6:30", "blocks": [
    tiles([(pct(P["overall_readmit"]), f"of {num(C['patients'])} diabetic patients were back within 30 days", None),
           (pct(KEY["target_pct"]), "had 2 or more other hospital stays in the past year", None),
           (pct(P["a1c_tested_pct"], 0), "had their blood sugar (A1C) checked during the stay", None),
           (pct(100 - P["high_a1c_changed_pct"], 0), "of patients with a high A1C had no change to their diabetes medications during the stay", None)]),
    cols(
        chart("hbars", "Prior hospital stays drive readmission",
              [{"label": "None" if r["level"] == "0" else f"{r['level']} prior stay" + ("" if r["level"] == "1" else "s"), "value": r["readmit_pct"],
                "tip": f"<b>{r['level']} prior stays</b><br>{pct(r['readmit_pct'])} readmitted<br>{num(r['patients'])} patients ({pct(r['pct_of_patients'])})"} for r in P["by"]["Prior hospital stays (past year)"]],
              {"labelW": 120, "max": 30, "ticks": [0, 10, 20, 30], "tickFmt": {"suffix": "%"}, "valFmt": {"d": 1, "suffix": "%"}, "ref": P["overall_readmit"]},
              sub="% readmitted within 30 days · dashed line = all patients"),
        chart("hbars", "So does where patients go next",
              [{"label": {"Skilled nursing / care facility": "Nursing / care facility", "Left against medical advice": "Left against advice", "Transferred to another hospital": "Other hospital"}.get(r["level"], r["level"]),
                "value": r["readmit_pct"], "tip": f"<b>{r['level']}</b><br>{pct(r['readmit_pct'])} readmitted<br>{num(r['patients'])} patients"}
               for r in sorted(P["by"]["Discharged to"], key=lambda r: -r["readmit_pct"]) if r["level"] != "Other / unknown"],
              {"labelW": 170, "max": 20, "ticks": [0, 10, 20], "tickFmt": {"suffix": "%"}, "valFmt": {"d": 1, "suffix": "%"}, "ref": P["overall_readmit"]},
              sub="% readmitted within 30 days, by discharge destination"),
    ),
    table([("Readmission model", False), ("Ranking accuracy (AUC; 0.5 = coin flip)", True), ("Top 10% flagged: readmitted vs. average", True)],
          [[r["model"] + (' <span class="chip">best</span>' if r is best else ""), f"{r['ROC-AUC']:.2f}", f"{r['Top-decile lift']:.1f}×"] for r in M["comparison"]],
          highlight=M["comparison"].index(best)),
    say(f"History and destination predict a return far better than diagnosis: 3+ prior stays means {pct(p3['readmit_pct'])} readmitted, and a nursing-facility discharge {pct(snf['readmit_pct'])} vs. {pct(home['readmit_pct'])} home. Three models reach only {min(r['ROC-AUC'] for r in M['comparison']):.2f}–{best['ROC-AUC']:.2f} accuracy; the neural network adds nothing. The clearest, most usable signal is the simplest one: prior hospital stays. And only {pct(P['a1c_tested_pct'], 0)} had an A1C test, so the hospital stay is a missed moment to review treatment."),
]}
SECTIONS.append(partA); preview(partA)

## 5 · A: Prior hospital stays are the clearest signal: 26.5% readmitted with 3+ vs. 8.1% with none  
*4:45 – 6:30*

- **9.0%** of 69,987 diabetic patients were back within 30 days
- **3.4%** had 2 or more other hospital stays in the past year
- **18%** had their blood sugar (A1C) checked during the stay
- **40%** of patients with a high A1C had no change to their diabetes medications during the stay

**Chart · Prior hospital stays drive readmission** — *% readmitted within 30 days · dashed line = all patients*

| label          |   value |
|:---------------|--------:|
| None           |     8.1 |
| 1 prior stay   |    12.9 |
| 2 prior stays  |    18.5 |
| 3+ prior stays |    26.5 |

**Chart · So does where patients go next** — *% readmitted within 30 days, by discharge destination*

| label                   |   value |
|:------------------------|--------:|
| Nursing / care facility |    15.1 |
| Other hospital          |    13.8 |
| Home with home health   |     9.5 |
| Left against advice     |     9.5 |
| Home                    |     6.9 |

| Readmission model       |   Ranking accuracy (AUC; 0.5 = coin flip) | Top 10% flagged: readmitted vs. average   |
|:------------------------|------------------------------------------:|:------------------------------------------|
| Penalized logistic (L1) |                                      0.64 | 2.0×                                      |
| Random forest best      |                                      0.66 | 2.3×                                      |
| Neural network (MLP)    |                                      0.64 | 2.0×                                      |

> **Say:** History and destination predict a return far better than diagnosis: 3+ prior stays means 26.5% readmitted, and a nursing-facility discharge 15.1% vs. 6.9% home. Three models reach only 0.64–0.66 accuracy; the neural network adds nothing. The clearest, most usable signal is the simplest one: prior hospital stays. And only 18% had an A1C test, so the hospital stay is a missed moment to review treatment.

## Slide 6 · B · The drugs and their price (6:30 – 8:00)

Notebook 03.

In [10]:
bc = L["brand_costs"]
partB = {"id": "partB", "nav": "6 · B: Price", "title": f"6 · B: The newer drugs went from {pct(KEY['newer_claims_13'])} to {pct(KEY['newer_claims_24'], 0)} of prescriptions; price is the payer's objection", "time": "6:30 – 8:00", "blocks": [
    chart("lines", "GLP-1 + SGLT2 share of Medicare diabetes drugs, 2013–2024 (same CMS file every year)",
          [{"name": "Share of spending", "color": "s2", "pts": [[t["year"], t["newer_share_cost"]] for t in L["trend"]], "end": pct(trend[2024]["newer_share_cost"], 0)},
           {"name": "Share of prescriptions", "color": "s1", "pts": [[t["year"], t["newer_share_claims"]] for t in L["trend"]], "end": pct(trend[2024]["newer_share_claims"], 0)}],
          {"H": 250, "ymax": 100, "yticks": [0, 25, 50, 75, 100], "yFmt": {"suffix": "%"}, "xticks": [2013, 2016, 2019, 2022, 2024], "xFmt": {"plain": True}, "tipFmt": {"d": 1, "suffix": "%"}},
          legend=[{"label": "Share of spending", "color": "s2"}, {"label": "Share of prescriptions (claims)", "color": "s1"}]),
    chart("pairbars", "A year of treatment: 2024 gross Medicare cost vs. negotiated price",
          [{"label": b["Brnd_Name"] + f" ({b['cls']})", "values": [b["gross_year"], b["negotiated_year"] if b["negotiated_year"] == b["negotiated_year"] else 0],
            "tip": f"<b>{b['Brnd_Name']}</b><br>2024 gross: ${b['gross_30d']:,.0f} per 30 days, ${b['gross_year']:,.0f} a year" + (f"<br>Negotiated: ${b['negotiated_30d']:,.0f} per 30 days" if b['negotiated_year'] == b['negotiated_year'] else "<br>No negotiated price yet")}
           for b in bc],
          {"max": 14000, "ticks": [0, 5000, 10000], "tickFmt": {"prefix": "$", "div": 1000, "suffix": "K"}, "valFmt": {"prefix": "$", "div": 1000, "d": 1, "suffix": "K"},
           "colors": ["s4", "s1"], "names": ["2024 gross", "Negotiated"]},
          legend=[{"label": "2024 gross (CMS: cost ÷ 30-day fills × 12)", "color": "s4"}, {"label": "Medicare negotiated price × 12", "color": "s1"}],
          sub="Trulicity and Mounjaro have no negotiated price yet; gross is before confidential rebates"),
    say(f"None of these drugs were used in the 1999–2008 hospital data. From one consistent Medicare source they went from {pct(KEY['newer_claims_13'])} of prescriptions in 2013 to {pct(KEY['newer_claims_24'], 0)} in 2024, while taking {pct(KEY['newer_cost_24'], 0)} of spending. A year of Jardiance costs Medicare about {money(L['annual_cost'][SG_G])} at 2024 gross cost and {money(L['annual_cost'][SG_N])} at the 2026 negotiated price, same drug, so it's a fair comparison. That price is the input to the next slide."),
]}
SECTIONS.append(partB); preview(partB)

## 6 · B: The newer drugs went from 1.5% to 33% of prescriptions; price is the payer's objection  
*6:30 – 8:00*

**Chart · GLP-1 + SGLT2 share of Medicare diabetes drugs, 2013–2024 (same CMS file every year)**

|      |   Share of spending |   Share of prescriptions |
|-----:|--------------------:|-------------------------:|
| 2013 |                 5.4 |                      1.5 |
| 2014 |                 6.7 |                      2.1 |
| 2015 |                 9.3 |                      3.2 |
| 2016 |                12.5 |                      4.3 |
| 2017 |                16.5 |                      5.5 |
| 2018 |                20.9 |                      7   |
| 2019 |                26.6 |                      9.2 |
| 2020 |                33.6 |                     11.7 |
| 2021 |                41.5 |                     15.2 |
| 2022 |                50.9 |                     20   |
| 2023 |                60.9 |                     27.4 |
| 2024 |                76   |                     33.3 |

**Chart · A year of treatment: 2024 gross Medicare cost vs. negotiated price** — *Trulicity and Mounjaro have no negotiated price yet; gross is before confidential rebates*

| label             |   2024 gross |   Negotiated |
|:------------------|-------------:|-------------:|
| Ozempic (GLP-1)   |        10840 |         3288 |
| Jardiance (SGLT2) |         6491 |         2364 |
| Mounjaro (GLP-1)  |        12475 |            0 |
| Trulicity (GLP-1) |        11381 |            0 |
| Farxiga (SGLT2)   |         6460 |         2142 |
| Rybelsus (GLP-1)  |        10989 |         3288 |

> **Say:** None of these drugs were used in the 1999–2008 hospital data. From one consistent Medicare source they went from 1.5% of prescriptions in 2013 to 33% in 2024, while taking 76% of spending. A year of Jardiance costs Medicare about $6,491 at 2024 gross cost and $2,364 at the 2026 negotiated price, same drug, so it's a fair comparison. That price is the input to the next slide.

## Slide 7 · C · Where it pays for itself (8:00 – 10:00)

Notebook 04, sections 1–3.

In [11]:
ORDER = [SG_N, SG_G]
grp_rows = [{"label": g["stays_group"], "values": [g[k] for k in ORDER],
             "tip": f"<b>{g['stays_group']}</b> ({pct(g['share_pct'])} of patients)<br>{g['prior_stays']:.1f} prior stays<br>" + "<br>".join(f"{k}: {money(g[k])}" for k in ORDER)}
            for g in V["groups"]]
partC = {"id": "partC", "nav": "7 · C: Value", "title": "7 · C: At the negotiated price, it pays for itself from two prior stays", "time": "8:00 – 10:00", "blocks": [
    p(f"**The model, per patient-year:** admissions avoided = prior hospital stays in the past year × 11% (EMPA-REG trial); savings = avoided × {money(KEY['cost'])}, Medicare's share of an average stay in 2026 dollars; net value = savings − the drug's annual cost. The stay the patient is leaving is **not** counted: a drug started at discharge can't prevent it. Only hospital savings count, so this is conservative."),
    table([("Price", False), ("Annual drug cost", True), ("Break-even: prior stays a year", True)],
          [[k, money(L["annual_cost"][k]), f"{V['breakeven'][k]:.2f}"] for k in ORDER]),
    chart("hgroup", "Net value per patient-year, by prior hospital stays in the past year", grp_rows,
          {"labelW": 120, "dom": [-7500, 7500], "ticks": [-5000, 0, 5000], "tickFmt": {"prefix": "$", "div": 1000, "suffix": "K"},
           "valFmt": {"prefix": "$", "div": 1000, "d": 1, "suffix": "K"}, "colors": ["s1", "s4"], "names": ORDER},
          legend=[{"label": "SGLT2 (Jardiance) · 2026 negotiated", "color": "s1"}, {"label": "SGLT2 (Jardiance) · 2024 gross", "color": "s4"}],
          sub="Positive = covering the drug pays for itself through avoided hospital stays"),
    h3("Does the past predict the future? What actually happened after discharge"),
    table([("Prior stays", False), ("Share of patients", True), ("Readmitted within 30 days", True), ("Readmitted at any point later", True)],
          [[g["stays_group"], pct(g["share_pct"]), pct(g["readmit_30_pct"]), pct(g["later_readmit_pct"])] for g in V["groups"]]),
    say(f"Break-even at the negotiated price is {KEY['be']:.2f} prior stays a year. With none, the drug can't pay for itself through hospital savings; with one, it's slightly short ({money(KEY['one_prior_net'])} a year); with two or more, it's about {money(KEY['net'])}. And the past does predict the future: patients with 2+ prior stays were readmitted far more often after discharge. At 2024 gross prices, only patients with 3+ prior stays clear it."),
]}
SECTIONS.append(partC); preview(partC)

## 7 · C: At the negotiated price, it pays for itself from two prior stays  
*8:00 – 10:00*

**The model, per patient-year:** admissions avoided = prior hospital stays in the past year × 11% (EMPA-REG trial); savings = avoided × $17,094, Medicare's share of an average stay in 2026 dollars; net value = savings − the drug's annual cost. The stay the patient is leaving is **not** counted: a drug started at discharge can't prevent it. Only hospital savings count, so this is conservative.

| Price                     | Annual drug cost   |   Break-even: prior stays a year |
|:--------------------------|:-------------------|---------------------------------:|
| SGLT2 · negotiated (2026) | $2,364             |                             1.26 |
| SGLT2 · gross 2024        | $6,491             |                             3.45 |

**Chart · Net value per patient-year, by prior hospital stays in the past year** — *Positive = covering the drug pays for itself through avoided hospital stays*

| group          |   SGLT2 · negotiated (2026) |   SGLT2 · gross 2024 |
|:---------------|----------------------------:|---------------------:|
| No prior stay  |                       -2364 |                -6491 |
| 1 prior stay   |                        -484 |                -4611 |
| 2 prior stays  |                        1397 |                -2730 |
| 3+ prior stays |                        5059 |                  932 |

### Does the past predict the future? What actually happened after discharge

| Prior stays    | Share of patients   | Readmitted within 30 days   | Readmitted at any point later   |
|:---------------|:--------------------|:----------------------------|:--------------------------------|
| No prior stay  | 88.3%               | 8.1%                        | 38.1%                           |
| 1 prior stay   | 8.3%                | 12.9%                       | 55.8%                           |
| 2 prior stays  | 2.1%                | 18.5%                       | 67.4%                           |
| 3+ prior stays | 1.3%                | 26.5%                       | 77.3%                           |

> **Say:** Break-even at the negotiated price is 1.26 prior stays a year. With none, the drug can't pay for itself through hospital savings; with one, it's slightly short (−$484 a year); with two or more, it's about $2,766. And the past does predict the future: patients with 2+ prior stays were readmitted far more often after discharge. At 2024 gross prices, only patients with 3+ prior stays clear it.

## Slide 8 · C · The rule a payer can write (10:00 – 12:00)

Notebook 04, sections 4–7.

In [12]:
tier_rows = [[t["stays_group"], pct(t["share_pct"]), pct(t["later_readmit_pct"]),
              "—" if t["breakeven_price_month"] == 0 else f"<b>{money(t['breakeven_price_month'])}</b>", t["at the negotiated SGLT2 price"]] for t in V["tiers"]]
tor = V["tornado"]
partC2 = {"id": "rule", "nav": "8 · C: Rule", "title": "8 · C: A coverage rule, the price that makes it work, and how sure we are", "time": "10:00 – 12:00", "blocks": [
    table([("Prior stays", False), ("Share of patients", True), ("Readmitted later", True), ("Covering pays for itself up to (per month)", True), (f"At the ${KEY['neg_month']:.0f}/month negotiated price", False)], tier_rows),
    callout(f"*Type 2 diabetes with heart or kidney disease, 2 or more hospital stays in the past 12 months, and not already on a GLP-1 or SGLT2 drug → cover with no prior authorization or step therapy.* Pays for itself at a net price up to about {money(KEY['price_2'])} a month with 2 prior stays and {money(KEY['price_3'])} with 3+. Heart or kidney disease is there because it's the guideline indication and the trial's population; it doesn't change how often patients return.", "The coverage rule a plan could write"),
    chart("tornado", "How sure are we? Net value per patient-year for the 2+ group, one assumption at a time",
          [{"label": t["assumption"], "low": t["low"], "high": t["high"], "tip": f"<b>{t['assumption']}</b><br>low: {money(t['low'])}<br>high: {money(t['high'])}"} for t in tor],
          {"labelW": 260, "dom": [-2000, 7000], "ticks": [0, 2000, 4000, 6000], "tickFmt": {"prefix": "$", "div": 1000, "d": 1, "suffix": "K"}, "base": KEY["net"]},
          sub="Orange = can turn negative"),
    say(f"Price is the strongest lever: each tier has a net price below which covering pays for itself. The case holds if these patients return half as often ({money(KEY['s_half'])}), for Medicare-insured patients ({money(KEY['medicare_net'])}) and for patients with heart disease ({money(KEY['heart_net'])}). It fails at the trial's lowest effect ({money(KEY['s_effect'])}) and at gross price ({money(KEY['s_gross'])}). For GLP-1 drugs I didn't assume the SGLT2 effect: at their negotiated price they'd need about a {KEY['glp1_2']:.0f}% reduction in hospital stays to pay off for this group."),
]}
SECTIONS.append(partC2); preview(partC2)

## 8 · C: A coverage rule, the price that makes it work, and how sure we are  
*10:00 – 12:00*

| Prior stays    | Share of patients   | Readmitted later   | Covering pays for itself up to (per month)   | At the $197/month negotiated price   |
|:---------------|:--------------------|:-------------------|:---------------------------------------------|:-------------------------------------|
| No prior stay  | 88.3%               | 38.1%              | —                                            | costs more than it saves             |
| 1 prior stay   | 8.3%                | 55.8%              | $157                                         | costs more than it saves             |
| 2 prior stays  | 2.1%                | 67.4%              | $313                                         | pays for itself                      |
| 3+ prior stays | 1.3%                | 77.3%              | $619                                         | pays for itself                      |

> **The coverage rule a plan could write** *Type 2 diabetes with heart or kidney disease, 2 or more hospital stays in the past 12 months, and not already on a GLP-1 or SGLT2 drug → cover with no prior authorization or step therapy.* Pays for itself at a net price up to about $313 a month with 2 prior stays and $619 with 3+. Heart or kidney disease is there because it's the guideline indication and the trial's population; it doesn't change how often patients return.

**Chart · How sure are we? Net value per patient-year for the 2+ group, one assumption at a time** — *Orange = can turn negative*

| assumption                           |   low |   high |
|:-------------------------------------|------:|-------:|
| Hospitalization reduction (4% – 18%) |  -499 |   6030 |
| Price (negotiated – 2024 gross)      | -1361 |   2766 |
| Cost of a stay ($9,664 – $20,654)    |   536 |   3834 |
| Future stays (50% – 100% of past)    |   201 |   2766 |

> **Say:** Price is the strongest lever: each tier has a net price below which covering pays for itself. The case holds if these patients return half as often ($201), for Medicare-insured patients ($2,715) and for patients with heart disease ($2,694). It fails at the trial's lowest effect (−$499) and at gross price (−$1,361). For GLP-1 drugs I didn't assume the SGLT2 effect: at their negotiated price they'd need about a 7% reduction in hospital stays to pay off for this group.

## Slide 9 · Recommendation (12:00 – 13:45)

Built around the payer, since the payer controls access.

In [13]:
reco = {"id": "reco", "nav": "9 · Recommendation", "title": "9 · Recommendation: win access through the payer", "time": "12:00 – 13:45", "blocks": [
    table([("#", False), ("Action", False), ("Who acts", False)], [
        ["1", "<b>Take the cost-offset case to Medicare Advantage plans and ACOs</b>: the organizations that pay for both the drug and the hospital stay, so they capture the saving. (In traditional Medicare the drug is paid by Part D and the stay by Part A, so no single party sees both.)", "Market access"],
        ["2", f"<b>Propose the coverage rule</b>: type 2 diabetes with heart or kidney disease, 2+ hospital stays in the past 12 months, not already on a GLP-1 or SGLT2 → no prior authorization, no step therapy, lower copay tier.", "Market access with the plan"],
        ["3", f"<b>Price by tier, backed by outcomes</b>: a net price at or below about {money(KEY['price_2'])} a month for the 2-stay tier, with a rebate if the group's admissions don't fall.", "Market access"],
        ["4", "<b>Give the plan the claims rule</b> so its care managers can flag eligible members and contact their doctors. It runs in any claims system, no model needed.", "The plan"],
        ["5", f"<b>Focus medical education on the discharge moment</b>: hospitalists and cardiologists see these patients in hospital, where only {pct(P['a1c_tested_pct'], 0)} had an A1C test. Trial evidence on reducing hospitalizations, on-label.", "Medical affairs"],
        ["6", "<b>Pilot with one plan for 12 months</b>: compare admissions for covered members against matched members, then use the result with the next plan.", "Market access with the plan"],
    ]),
    say("The doctor still decides, but the payer decides whether the doctor's decision reaches the patient. These six steps move the access decision, with a price and a rule the payer can check."),
]}
SECTIONS.append(reco); preview(reco)

## 9 · Recommendation: win access through the payer  
*12:00 – 13:45*

|   # | Action                                                                                                                                                                                                                                                                     | Who acts                    |
|----:|:---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------------------|
|   1 | Take the cost-offset case to Medicare Advantage plans and ACOs: the organizations that pay for both the drug and the hospital stay, so they capture the saving. (In traditional Medicare the drug is paid by Part D and the stay by Part A, so no single party sees both.) | Market access               |
|   2 | Propose the coverage rule: type 2 diabetes with heart or kidney disease, 2+ hospital stays in the past 12 months, not already on a GLP-1 or SGLT2 → no prior authorization, no step therapy, lower copay tier.                                                             | Market access with the plan |
|   3 | Price by tier, backed by outcomes: a net price at or below about $313 a month for the 2-stay tier, with a rebate if the group's admissions don't fall.                                                                                                                     | Market access               |
|   4 | Give the plan the claims rule so its care managers can flag eligible members and contact their doctors. It runs in any claims system, no model needed.                                                                                                                     | The plan                    |
|   5 | Focus medical education on the discharge moment: hospitalists and cardiologists see these patients in hospital, where only 18% had an A1C test. Trial evidence on reducing hospitalizations, on-label.                                                                     | Medical affairs             |
|   6 | Pilot with one plan for 12 months: compare admissions for covered members against matched members, then use the result with the next plan.                                                                                                                                 | Market access with the plan |

> **Say:** The doctor still decides, but the payer decides whether the doctor's decision reaches the patient. These six steps move the access decision, with a price and a rule the payer can check.

## Slide 10 · Risks and next steps (13:45 – 15:00)

In [14]:
LIMITS = [
    "Hospital patterns come from 1999–2008; costs are in 2026 dollars, but who gets readmitted today may differ.",
    "The 11% effect comes from EMPA-REG (outpatients with heart disease; time to first hospitalization), applied to admission counts; tested at 4%.",
    "Next year's stays are assumed to match last year's; tested at half.",
    "Only hospital savings count, not deaths, kidney failure or quality of life, so value is understated.",
    "Gross cost is before confidential rebates; the true net price sits between gross and negotiated.",
    "For GLP-1 drugs the deck shows the effect they would need rather than assuming one.",
]
NEXT = [
    "Re-estimate prior and future stays with recent Medicare claims.",
    "Add the heart-failure and kidney benefits, which are larger than the all-cause hospitalization effect.",
    "Size the national opportunity from the number of diabetic discharges per year.",
    "Run the pilot with matched controls to measure the effect directly.",
]
risks = {"id": "risks", "nav": "10 · Risks", "title": "10 · Risks, limitations and next steps", "time": "13:45 – 15:00", "blocks": [
    h3("How the years are reconciled"),
    table([("Input", False), ("Data year", False), ("How it's aligned", False), ("Effect on the result", False)],
          [[t["input"], t["data_year"], t["how_aligned"], t["effect_on_result"]] for t in V["time_table"]]),
    {"type": "limits", "cols": [{"title": "Limitations", "md": LIMITS, "items": [md(x) for x in LIMITS]},
                                {"title": "Next steps", "md": NEXT, "items": [md(x) for x in NEXT]}]},
    say("The direction holds: the drug pays for itself for frequent hospital users at negotiated prices, and the payer is who can act on that. The exact dollars depend on assumptions I've laid out and tested; the next step is recent claims data and a pilot."),
]}
SECTIONS.append(risks); preview(risks)

## 10 · Risks, limitations and next steps  
*13:45 – 15:00*

### How the years are reconciled

| Input                            | Data year                      | How it's aligned                                                                                     | Effect on the result                                                                                |
|:---------------------------------|:-------------------------------|:-----------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------|
| Prior hospital stays per patient | 1999–2008                      | Checked against what happened after discharge (section 3) and recent readmission rates; 50% scenario | Likely understates value: 30-day readmission here (11.4% of stays) is below recent studies (16–20%) |
| Drug effect (11%)                | 2015 trial                     | Range 4–18%; heart-disease subgroup tested                                                           | Could overstate: the trial measured first hospitalization in outpatients with heart disease         |
| Cost of a stay                   | 2024 (CMS); 2018 (HCUP check)  | Medicare's share, moved to 2026 with the BLS hospital price index                                    | CPI may run above Medicare's own rate updates; low case uses no inflation                           |
| Drug prices                      | 2024 gross; 2026/27 negotiated | Negotiated prices are already the decision years                                                     | Gross is before confidential rebates, so it overstates the net price                                |

**Limitations**
- Hospital patterns come from 1999–2008; costs are in 2026 dollars, but who gets readmitted today may differ.
- The 11% effect comes from EMPA-REG (outpatients with heart disease; time to first hospitalization), applied to admission counts; tested at 4%.
- Next year's stays are assumed to match last year's; tested at half.
- Only hospital savings count, not deaths, kidney failure or quality of life, so value is understated.
- Gross cost is before confidential rebates; the true net price sits between gross and negotiated.
- For GLP-1 drugs the deck shows the effect they would need rather than assuming one.

**Next steps**
- Re-estimate prior and future stays with recent Medicare claims.
- Add the heart-failure and kidney benefits, which are larger than the all-cause hospitalization effect.
- Size the national opportunity from the number of diabetic discharges per year.
- Run the pilot with matched controls to measure the effect directly.

> **Say:** The direction holds: the drug pays for itself for frequent hospital users at negotiated prices, and the payer is who can act on that. The exact dollars depend on assumptions I've laid out and tested; the next step is recent claims data and a pilot.

## Build the page

In [15]:
FOOTER = ["Built from five Jupyter notebooks in <span class=\"mono\">avaitt/personal-project</span>. Every number is read from the notebooks' saved results or the sourced assumptions file.",
          "Data: Kaggle Diabetes 130-US Hospitals (Strack et al., 2014) · CMS Medicare Part D (2013–2024) · CMS Medicare inpatient payments (2024) · BLS CPI · CMS negotiated prices · EMPA-REG OUTCOME · AHRQ HCUP (cross-check) · NLM RxClass · FDA approval dates."]
PAGE = {**HERO, "sections": SECTIONS, "footer": FOOTER}
blob = json.dumps(PAGE, ensure_ascii=False)
assert all(s["title"] and s["time"] and s["blocks"] for s in SECTIONS)
assert "NaN" not in blob, "a missing value reached the page"
print("Section times:", [s["time"] for s in SECTIONS])
template = (ROOT / "site" / "story_template.html").read_text()
assert template.count("/*__PAGE__*/{}") == 1
out = ROOT / "docs" / "story.html"
out.write_text(template.replace("/*__PAGE__*/{}", json.dumps(PAGE, ensure_ascii=False, separators=(",", ":"))))
print(f"wrote {out.relative_to(ROOT)} ({out.stat().st_size / 1024:.0f} KB, {len(SECTIONS)} sections)")

Section times: ['0:30 – 1:30', '1:30 – 2:45', '2:45 – 3:45', '3:45 – 4:15', '4:15 – 4:45', '4:45 – 6:30', '6:30 – 8:00', '8:00 – 10:00', '10:00 – 12:00', '12:00 – 13:45', '13:45 – 15:00']
wrote docs/story.html (56 KB, 11 sections)
